In [ ]:
# Safe setup block
import warnings
warnings.filterwarnings('ignore')

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'


In [ ]:
!pip install -q tensorflow keras-tuner torch torchvision torchaudio nlpaug augly wandb imbalanced-learn keras_cv seaborn matplotlib numpy pandas scikit-learn

In [ ]:
import sys
import tensorflow as tf
import torch

print(f"Python version: {sys.version}")
print(f"TensorFlow version: {tf.__version__}")
print(f"PyTorch version: {torch.__version__}")

print("\n--- GPU Availability ---")
print(f"TF GPU: {len(tf.config.list_physical_devices('GPU')) > 0}")
print(f"PyTorch GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"PyTorch Device Name: {torch.cuda.get_device_name(0)}")


In [ ]:
# --- INLINED UTILS FUNCTION ---
import os
import random
import numpy as np
import tensorflow as tf
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for academic-grade visualizations
sns.set_theme(style="whitegrid", context="talk", palette="deep")

def set_seed(seed=42):
    """
    Ensures absolute reproducibility across all stochastic operations.
    Sets seeds for Python random, NumPy, TensorFlow, and PyTorch.
    """
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # TensorFlow
    tf.random.set_seed(seed)
    
    # PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    print(f"[Utils] Random seed globally set to {seed} for reproducibility.")

def plot_tf_history(history, title="Training History"):
    """
    Plots the accuracy and loss curves for a Keras/TensorFlow training history.
    """
    acc = history.history.get('accuracy', history.history.get('acc', []))
    val_acc = history.history.get('val_accuracy', history.history.get('val_acc', []))
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(1, len(acc) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(title, fontsize=20, fontweight='bold', y=1.05)

    # Accuracy Plot
    ax1.plot(epochs, acc, 'bo-', label='Training Acc', alpha=0.8, linewidth=2)
    ax1.plot(epochs, val_acc, 'ro-', label='Validation Acc', alpha=0.8, linewidth=2)
    ax1.set_title('Accuracy')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Accuracy')
    ax1.legend()

    # Loss Plot
    ax2.plot(epochs, loss, 'bo-', label='Training Loss', alpha=0.8, linewidth=2)
    ax2.plot(epochs, val_loss, 'ro-', label='Validation Loss', alpha=0.8, linewidth=2)
    ax2.set_title('Loss')
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Loss')
    ax2.legend()

    plt.tight_layout()
    plt.show()

def plot_torch_history(train_losses, val_losses, train_acc=None, val_acc=None, title="PyTorch Training History"):
    """
    Plots the loss (and optionally accuracy) curves for PyTorch manual training loops.
    """
    epochs = range(1, len(train_losses) + 1)
    
    if train_acc is not None and val_acc is not None:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
        fig.suptitle(title, fontsize=20, fontweight='bold', y=1.05)

        # Accuracy Plot
        ax1.plot(epochs, train_acc, 'bo-', label='Training Acc', alpha=0.8, linewidth=2)
        ax1.plot(epochs, val_acc, 'ro-', label='Validation Acc', alpha=0.8, linewidth=2)
        ax1.set_title('Accuracy')
        ax1.set_xlabel('Epochs')
        ax1.set_ylabel('Accuracy')
        ax1.legend()

        # Loss Plot
        ax2.plot(epochs, train_losses, 'bo-', label='Training Loss', alpha=0.8, linewidth=2)
        ax2.plot(epochs, val_losses, 'ro-', label='Validation Loss', alpha=0.8, linewidth=2)
        ax2.set_title('Loss')
        ax2.set_xlabel('Epochs')
        ax2.set_ylabel('Loss')
        ax2.legend()
    else:
        plt.figure(figsize=(8, 6))
        plt.plot(epochs, train_losses, 'bo-', label='Training Loss', alpha=0.8, linewidth=2)
        plt.plot(epochs, val_losses, 'ro-', label='Validation Loss', alpha=0.8, linewidth=2)
        plt.title(f"{title} - Loss", fontsize=16, fontweight='bold')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend()
        
    plt.tight_layout()
    plt.show()

def compare_models_tf(histories_dict, metric='val_accuracy', title="Model Comparison"):
    """
    Overlays multiple Keras histories on a single plot for A/B testing comparison.
    histories_dict: dict of { 'Label': history_object }
    """
    plt.figure(figsize=(10, 7))
    plt.title(title, fontsize=18, fontweight='bold')
    
    for label, history in histories_dict.items():
        data = history.history.get(metric, history.history.get('val_acc', []))
        epochs = range(1, len(data) + 1)
        plt.plot(epochs, data, marker='o', label=label, linewidth=2, alpha=0.8)
        
    plt.xlabel('Epochs')
    plt.ylabel(metric.replace('_', ' ').title())
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()


# 07. Document Image Augmentation: Bleed-through and Blur

## 1. Problem Definition
OCR models often fail on historic or poorly scanned documents. We simulate physical degradation.

## 2. Techniques
- **Bleed-through**: Simulating ink from the reverse page.
- **Motion Blur**: Simulating scanner movement.
- **Binarization Noise**: Adding salt and pepper noise simulating thresholding errors.